# Advanced document indexing

## What this notebook does

This notebook explores advanced indexing techniques for Retrieval-Augmented Generation (RAG). It focuses on how document splitting and embedding strategies affect retrieval quality, especially when the source material comes from structured web pages.

The first part compares granular HTML-section chunks with coarser text chunks. Later sections use parent-document and multi-vector retrievers to preserve useful context while still searching over compact, targeted representations such as child chunks, summaries, hypothetical questions, and expanded chunks.

The notebook is configured from the project-root `.env` file and can run with OpenAI, Ollama, Gemini, or DeepSeek for chat. Embeddings can use OpenAI, Ollama, or Gemini; with DeepSeek chat they default to Gemini.


In [ ]:
# Select the LLM provider: "openai", "ollama", "gemini", or "deepseek".
# Values are loaded from the project-root .env file when present.
import os
import getpass
from pathlib import Path

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings


class ChatDeepSeek(ChatOpenAI):
    """Use function calling for structured output; DeepSeek rejects json_schema."""

    def with_structured_output(
        self, schema=None, *, method="function_calling",
        include_raw=False, strict=None, **kwargs
    ):
        return super().with_structured_output(
            schema, method=method, include_raw=include_raw,
            strict=strict, **kwargs
        )


def load_project_env(filename=".env", override=True):
    current = Path.cwd().resolve()
    for directory in [current, *current.parents]:
        env_path = directory / filename
        if env_path.exists():
            for line in env_path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip('"').strip("'")
                if override or key not in os.environ:
                    os.environ[key] = value
            return env_path
    return None


PROJECT_ENV_PATH = load_project_env(override=True)

# LangSmith tracing is configured before any LangChain objects are created.
os.environ.setdefault("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
os.environ.setdefault("LANGSMITH_PROJECT", "advanced-indexing")
if os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = os.getenv("LANGSMITH_TRACING", "true")

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "ollama").strip().lower()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e2b")
OLLAMA_EMBEDDING_MODEL = os.getenv("OLLAMA_EMBEDDING_MODEL", "nomic-embed-text")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-flash-latest")
GEMINI_EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-2-preview")

DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-v4-pro")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")
DEEPSEEK_THINKING = os.getenv("DEEPSEEK_THINKING", "disabled")

SUPPORTED_LLM_PROVIDERS = {"openai", "ollama", "gemini", "deepseek"}
if LLM_PROVIDER not in SUPPORTED_LLM_PROVIDERS:
    raise ValueError(f"Unsupported LLM_PROVIDER: {LLM_PROVIDER}. Use one of: {sorted(SUPPORTED_LLM_PROVIDERS)}")

# DeepSeek provides chat models but no embeddings API.
default_embedding_provider = "gemini" if LLM_PROVIDER == "deepseek" else LLM_PROVIDER
EMBEDDING_PROVIDER = os.getenv("EMBEDDING_PROVIDER") or default_embedding_provider
EMBEDDING_PROVIDER = EMBEDDING_PROVIDER.strip().lower()
SUPPORTED_EMBEDDING_PROVIDERS = {"openai", "ollama", "gemini"}
if EMBEDDING_PROVIDER not in SUPPORTED_EMBEDDING_PROVIDERS:
    raise ValueError(f"Unsupported EMBEDDING_PROVIDER: {EMBEDDING_PROVIDER}. Use one of: {sorted(SUPPORTED_EMBEDDING_PROVIDERS)}")


def _get_api_key(*names):
    for name in names:
        value = os.getenv(name)
        if value:
            return value
    return getpass.getpass(f"Enter your {names[0]}")


class GeminiEmbeddingsOneByOne:
    def __init__(self, embeddings):
        self.embeddings = embeddings

    def embed_documents(self, texts):
        # Gemini embedding models can return one vector per batch with this
        # LangChain version. Chroma expects one vector per document, so embed
        # each chunk separately.
        return [
            self.embeddings.embed_documents([text], batch_size=1)[0]
            for text in texts
        ]

    def embed_query(self, text):
        return self.embeddings.embed_query(text)


def ensure_ollama_model_available(model_name):
    import ollama

    try:
        installed_models = {model.model for model in ollama.list().models}
    except Exception as exc:
        raise RuntimeError(
            f"Could not connect to Ollama at {OLLAMA_BASE_URL}. "
            "Start Ollama before running this notebook."
        ) from exc

    model_aliases = {name.removesuffix(":latest") for name in installed_models}
    if model_name not in installed_models and model_name not in model_aliases:
        raise RuntimeError(
            f'Ollama model "{model_name}" is not installed. '
            f'Run `ollama pull {model_name}` in a terminal, then restart the notebook kernel.'
        )


def get_embeddings_model():
    if EMBEDDING_PROVIDER == "openai":
        return OpenAIEmbeddings(
            model=OPENAI_EMBEDDING_MODEL,
            openai_api_key=_get_api_key("OPENAI_API_KEY"),
        )
    if EMBEDDING_PROVIDER == "ollama":
        ensure_ollama_model_available(OLLAMA_EMBEDDING_MODEL)
        return OllamaEmbeddings(
            model=OLLAMA_EMBEDDING_MODEL,
            base_url=OLLAMA_BASE_URL,
        )
    if EMBEDDING_PROVIDER == "gemini":
        embeddings = GoogleGenerativeAIEmbeddings(
            model=GEMINI_EMBEDDING_MODEL,
            google_api_key=_get_api_key("GEMINI_API_KEY", "GOOGLE_API_KEY"),
        )
        return GeminiEmbeddingsOneByOne(embeddings)


def get_chat_model():
    if LLM_PROVIDER == "openai":
        return ChatOpenAI(
            model=OPENAI_MODEL,
            openai_api_key=_get_api_key("OPENAI_API_KEY"),
        )
    if LLM_PROVIDER == "ollama":
        ensure_ollama_model_available(OLLAMA_MODEL)
        return ChatOllama(
            model=OLLAMA_MODEL,
            base_url=OLLAMA_BASE_URL,
        )
    if LLM_PROVIDER == "gemini":
        return ChatGoogleGenerativeAI(
            model=GEMINI_MODEL,
            google_api_key=_get_api_key("GEMINI_API_KEY", "GOOGLE_API_KEY"),
        )
    if LLM_PROVIDER == "deepseek":
        return ChatDeepSeek(
            model=DEEPSEEK_MODEL,
            openai_api_key=_get_api_key("DEEPSEEK_API_KEY"),
            base_url=DEEPSEEK_BASE_URL,
            extra_body={"thinking": {"type": DEEPSEEK_THINKING}},
        )


def message_text(message):
    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and "text" in item:
                parts.append(str(item["text"]))
            else:
                parts.append(str(item))
        return "\n".join(parts)
    return str(content)


print(f"Loaded env from: {PROJECT_ENV_PATH or 'not found'}")
print(f"Using {LLM_PROVIDER} chat model with {EMBEDDING_PROVIDER} embeddings")
if LLM_PROVIDER == "ollama":
    print(f"Ollama chat model: {OLLAMA_MODEL}")
elif LLM_PROVIDER == "gemini":
    print(f"Gemini chat model: {GEMINI_MODEL}")
elif LLM_PROVIDER == "deepseek":
    print(f"DeepSeek chat model: {DEEPSEEK_MODEL}")
print(f"LangSmith tracing: {os.getenv('LANGSMITH_TRACING', 'false')} | project: {os.getenv('LANGSMITH_PROJECT')}")


### Why split by HTML headers

This section uses Wikivoyage pages to study how splitting granularity changes retrieval behavior. HTML pages already contain structural signals such as `h1` and `h2` headers, so the notebook first uses those headers to create smaller, topic-oriented chunks.

The goal is to compare detailed section-level retrieval with larger, coarser chunks. Granular chunks can match specific facts more precisely, while coarse chunks may preserve more surrounding context.

> **Local data.** The Wikivoyage pages are read from the saved copies in `data/wikivoyage/` through `local_html.LocalHtmlLoader`, so this notebook needs no internet access for ingestion. The class mirrors `AsyncHtmlLoader`: it returns the raw HTML of each page and keeps the original URL as `source` metadata.


# Splitting and ingesting HTML content

## Splitting and ingesting the content of a single URL (on Cornwall)

### Preparing the Chroma DB collections

In [ ]:
from langchain_chroma import Chroma


### Chroma collections for granular and coarse chunks

Two Chroma collections are created for the same Cornwall source page. One collection stores granular chunks generated from HTML sections, and the other stores coarser chunks generated after converting the page to plain text.

Resetting each collection keeps the experiment repeatable: every notebook run starts from an empty vector store, so search results reflect only the chunks inserted by the current cells.


In [ ]:
corwnall_granular_collection = Chroma( #A
    collection_name="cornwall_granular",
    embedding_function=get_embeddings_model(),
)

corwnall_granular_collection.reset_collection() #B
# A Create a Chorma DB collection
# B Reset the collection in case it already exists 

In [ ]:
corwnall_coarse_collection = Chroma( #A 
    collection_name="cornwall_coarse",
    embedding_function=get_embeddings_model(),
)

corwnall_coarse_collection.reset_collection() #B
# A Create a Chorma DB collection
# B Reset the collection in case it already exists 

### Loading the HTML content with the LocalHtmlLoader 

### Loading the Cornwall page

`LocalHtmlLoader` reads the saved Wikivoyage page for Cornwall. The loaded HTML document is the shared input for both indexing strategies, which makes the later comparison fair: only the chunking strategy changes, not the source content.


In [ ]:
from local_html import LocalHtmlLoader

In [ ]:
destination_url = "https://en.wikivoyage.org/wiki/Cornwall"

In [ ]:
html_loader = LocalHtmlLoader(destination_url)

In [ ]:
docs = html_loader.load()

In [ ]:
len(docs)

### Splitting into granular chunks with the HTMLSectionSplitter 

### Granular chunks from HTML sections

`HTMLSectionSplitter` uses selected HTML headers to split the page into section-level documents. Here the split is based on `h1` and `h2`, which usually correspond to meaningful page sections.

These chunks are useful when the query asks for specific local information, such as events or festivals, because retrieval can return a focused section instead of a long mixed-content passage.


In [ ]:
from langchain_text_splitters import HTMLSectionSplitter

In [ ]:
headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(
    headers_to_split_on=headers_to_split_on)

In [ ]:
def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content #A
        temp_chunks = html_section_splitter.split_text(
            html_string) #B
        all_chunks.extend(temp_chunks) 

    return all_chunks

#A Extract the HTML text from the document
#B Each chunk is a H1 or H2 HTML section

In [ ]:
granular_chunks = split_docs_into_granular_chunks(docs)

In [ ]:
granular_chunks

#### Ingesting granular chunks

In [ ]:
corwnall_granular_collection.add_documents(documents=granular_chunks)

### Searching granular chunks

After inserting the granular chunks into Chroma, the notebook searches for events or festivals in Cornwall. This shows what the section-based index can retrieve when the question targets a specific topic.


#### Searching granular chunks

In [ ]:
results = corwnall_granular_collection.similarity_search(
    query="Events or festivals in Cornwall",k=3)
for doc in results:
    print(doc)

### Splitting into coarse chunks with the RecursiveCharacterTextSplitter 

### Coarse chunks from plain text

The coarse strategy first converts HTML to plain text and then applies `RecursiveCharacterTextSplitter` with larger chunks and overlap. This produces broader passages that retain more surrounding context.

Coarse chunks can be less precise than section-level chunks, but they may be better when the answer depends on neighboring information that would otherwise be split away.


In [ ]:
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
html2text_transformer = Html2TextTransformer()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000, chunk_overlap=300
)

In [ ]:
def split_docs_into_coarse_chunks(docs):
    text_docs = html2text_transformer.transform_documents(
        docs) #A 
    coarse_chunks = text_splitter.split_documents(
        text_docs)

    return coarse_chunks
#A transform HTML docs into clean text docs

In [ ]:
coarse_chunks = split_docs_into_coarse_chunks(docs)

#### Ingesting coarse chunks

In [ ]:
corwnall_coarse_collection.add_documents(documents=coarse_chunks)

### Searching coarse chunks

The same query is run against the coarse collection. Comparing these results with the granular search helps reveal the trade-off between precise retrieval and context preservation.


#### Searching coarse chunks

In [ ]:
results = corwnall_coarse_collection.similarity_search(
    query="Events or festivals in Cornwall",k=3)
for doc in results:
    print(doc)

### Scaling the experiment to multiple URLs

The notebook then repeats the same granular and coarse ingestion pattern across several UK Wikivoyage destinations. This creates a larger search space and makes the comparison closer to a real RAG knowledge base.

Reducing the destination list is a practical way to control processing time and embedding costs while experimenting with the indexing strategy.


## Splitting and ingesting the content of various URLs (across UK destinations)

### Preparing the Chroma DB collections

In [ ]:
uk_granular_collection = Chroma( #A
    collection_name="uk_granular",
    embedding_function=get_embeddings_model(),
)

uk_granular_collection.reset_collection() #B

In [ ]:
uk_coarse_collection = Chroma( #A
    collection_name="uk_coarse",
    embedding_function=get_embeddings_model(),
)

uk_coarse_collection.reset_collection() #B

### Splitting and ingesting HTML content with the HTMLSectionSplitter 

In [ ]:
# Reduce this list if you want to save on processing fees
uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall", 
    "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
    "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven",
    "East_Sussex", "Brighton", "Battle", "Hastings_(England)", 
    "Rye_(England)", "Seaford", "Ashdown_Forest"
] 

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"

In [ ]:
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' for d in uk_destinations]

In [ ]:
for destination_url in uk_destination_urls:
    html_loader = LocalHtmlLoader(destination_url) #C
    docs =  html_loader.load() #D
    
    for doc in docs:
        print(doc.metadata)
        granular_chunks = split_docs_into_granular_chunks(docs)
        uk_granular_collection.add_documents(documents=granular_chunks)

        coarse_chunks = split_docs_into_coarse_chunks(docs)
        uk_coarse_collection.add_documents(documents=coarse_chunks)
#A Create a Chroma DB collection
#B Reset the collection in case it already exists 
#C Loader for one destination
#D Documents of one destination 

### Comparing granular and coarse retrieval across destinations

The final searches in this section compare granular and coarse indexes across the multi-page collection. Trying different queries, such as events in East Sussex or beaches in Cornwall, helps tune the balance between detailed chunks and broader contextual chunks.


#### Searching 

In [ ]:
granular_results = uk_granular_collection.similarity_search(
    query="Events or festivals in East Sussex",k=4)
for doc in granular_results:
    print(doc)

In [ ]:
coarse_results = uk_coarse_collection.similarity_search(
    query="Events or festivals in East Sussex",k=4)
for doc in coarse_results:
    print(doc)

In [ ]:
granular_results = uk_granular_collection.similarity_search(
    query="Beaches in Conrwall",k=4)
for doc in granular_results:
    print(doc)

In [ ]:
coarse_results = uk_coarse_collection.similarity_search(
    query="Beaches in Cornwall",k=4)
for doc in coarse_results:
    print(doc)

### 8.4 Embedding strategy

The notebook now moves from splitting strategy to embedding strategy. Instead of storing only one vector per document chunk, the following sections show ways to store multiple vectors that point back to richer source content.

The common idea is a two-layer structure: smaller retrieval chunks or generated representations are embedded for precise search, while larger synthesis chunks are returned to the LLM so the answer has enough context.


# Embedding strategy

## Embedding child chunks with ParentDocumentRetriever

In [ ]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from local_html import LocalHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Setting up the Parent Document retriever

### ParentDocumentRetriever: child vectors, parent context

`ParentDocumentRetriever` addresses the chunk-size trade-off. Small child chunks are better for detailed semantic matching, but larger parent chunks are better as context for answer generation.

The retriever stores child chunk embeddings in Chroma and parent chunks in an in-memory document store. A match on a child chunk returns the parent document, giving the LLM the broader surrounding context.


In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
    embedding_function=get_embeddings_model(),
)

child_chunks_collection.reset_collection() #D

doc_store = InMemoryStore() #E

parent_doc_retriever = ParentDocumentRetriever( #F
    vectorstore=child_chunks_collection,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

### Ingesting the content into doc and vector store

### Ingesting parent and child chunks

Calling `parent_doc_retriever.add_documents()` performs both indexing steps: it creates parent chunks for synthesis, creates child chunks for retrieval, embeds the child chunks, and records the link back to the parent chunk.


In [ ]:
for destination_url in uk_destination_urls:
    html_loader = LocalHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(html_docs) #C

    print(f'Ingesting {destination_url}')
    parent_doc_retriever.add_documents(text_docs, ids=None) #D

#A Loader for destination web page
#B HTML documents of one destination 
#C Transform HTML docs into clean text deocs
#D Ingest coarse chunks into document store and granular chunks into vector store

In [ ]:
list(doc_store.yield_keys())
#A Show the keys of the added coarse chunks

### Performing a search on granular information 

### Searching through child chunks, returning parents

The query is matched against the child chunk vectors, but the returned document is the parent chunk. This is why the first result contains more travel context than a direct search against the child collection alone.


In [ ]:
retrieved_docs = parent_doc_retriever.invoke("Cornwall Ranger")

In [ ]:
len(retrieved_docs)

In [ ]:
retrieved_docs[0]

### Comparing with direct semantic search on child chunks

### Direct child search comparison

A direct Chroma search over child chunks can find the precise fact, but it returns only the small matched passage. Comparing it with the parent-document result shows why parent retrieval is useful for LLM synthesis.


In [ ]:
child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")

In [ ]:
len(child_docs_only)

In [ ]:
child_docs_only[0]

In [ ]:
# IMPORTANT: as you can see a granular search would have identified the chunk, but it would have lost the usefulcontext about travelling in Cornwall

### MultiVectorRetriever with child chunks

`MultiVectorRetriever` implements the same broad idea with more explicit control. The vector store contains many child vectors, while the byte store keeps the larger parent chunks keyed by a document id.

This gives more flexibility than `ParentDocumentRetriever` because the notebook controls exactly how child chunks are linked to parent chunks through metadata.


## Embedding child chunks with MultiVectorRetriever

In [ ]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from local_html import LocalHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

### Setting up the Multi vector retriever

In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
    embedding_function=get_embeddings_model(),
)

child_chunks_collection.reset_collection() #D

doc_byte_store = InMemoryByteStore() #E
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #F
    vectorstore=child_chunks_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

### Ingesting the content into doc and vector store

### Building child-to-parent links manually

The ingestion loop creates ids for each parent chunk, splits each parent into child chunks, writes the parent id into every child chunk's metadata, and then stores child chunks and parent chunks in their respective stores.


In [ ]:
for destination_url in uk_destination_urls:
    html_loader = LocalHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_granular_chunks = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E
        
        coarse_chunk_id = coarse_chunks_ids[i]
            
        granular_chunks = child_splitter.split_documents(
            [coarse_chunk]) #F

        for granular_chunk in granular_chunks:
            granular_chunk.metadata[doc_key] = coarse_chunk_id #G

        all_granular_chunks.extend(granular_chunks)

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_granular_chunks) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination 
#C transform HTML docs into clean text docs
#D Split the destination content into parent coarse chunks
#E Iterate over the parent coarse chunks
#F Create child granular chunks form each parent coarse chunk
#G Link each child granular chunk to its parent coarse chunk
#H Ingest the child granular chunks into the vector store
#I Ingest the parent coarse chunks into the document store

### Performing a search on granular information

### Searching with MultiVectorRetriever

As with `ParentDocumentRetriever`, the search is performed over granular child vectors while the retriever returns the larger parent document. This preserves context while still allowing precise semantic matching.


In [ ]:
retrieved_docs = multi_vector_retriever.invoke(
    "Cornwall Ranger")

In [ ]:
len(retrieved_docs)

In [ ]:
retrieved_docs[0]

In [ ]:
##IMPORTANT: same as Parent Document retriever, but more control and flexibility on how to link child to parent chunks

### Comparing with direct semantic search on child chunks

In [ ]:
child_docs_only =  child_chunks_collection.similarity_search(
    "Cornwall Ranger")

In [ ]:
len(child_docs_only)

In [ ]:
child_docs_only[0]

In [ ]:
## IMPORTANT: Same as before

### Summary embeddings

Large chunks can contain too much incidental text, which can make their embeddings noisy. Summary embeddings reduce that noise by representing each parent chunk with a concise LLM-generated summary.

The vector store indexes summaries, but the retriever still returns the original larger chunks. This keeps retrieval focused while preserving rich source context for answer generation.


## Embedding summaries with MultiVectorRetriever

In [ ]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from local_html import LocalHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid

### Setting up the Multi vector retriever (similar to when embedding child chunks)

In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000) #A

summaries_collection = Chroma( #B
    collection_name="uk_summaries",
    embedding_function=get_embeddings_model(),
)

summaries_collection.reset_collection() #C

doc_byte_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=summaries_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host parent coarse chunks
#E Retriever to link parent coarse chunks to child granular chunks

### Setting up the summarization chain

### Summarization chain

The summarization chain extracts `page_content`, prompts the selected chat model to summarize it, and parses the output into plain text. This is slower than child chunk indexing because every coarse chunk requires an LLM call.


In [ ]:
llm = get_chat_model()

In [ ]:
summarization_chain = (
    {"document": lambda x: x.page_content} #A
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{document}") #B
    | llm
    | StrOutputParser())

#A Grab the text content from the document
#B Instantiate a prompt asking to generate summary of the provided text
#C Send the LLM the instantiated prompt 
#D Extract the summary text from the response

### Ingesting the coarse chunks and related summaries into doc and vector store

### Ingesting summaries and source chunks

For each coarse chunk, the notebook generates a summary document linked to the original chunk id. The summary is embedded and stored in Chroma; the full coarse chunk is stored in the byte store for retrieval after a match.


In [ ]:
for destination_url in uk_destination_urls:
    html_loader = LocalHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_summaries = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E
        
        coarse_chunk_id = coarse_chunks_ids[i]
            
        summary_text =  summarization_chain.invoke(
            coarse_chunk) #F
        summary_doc = Document(page_content=summary_text, 
                               metadata={doc_key: coarse_chunk_id})

        all_summaries.append(summary_doc) #G

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_summaries) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination 
#C transform HTML docs into clean text docs
#D Split the destination content into coarse chunks
#E Iterate over the coarse chunks
#F Generate a summary for the coarse chunk thorugh the summarization chain
#G Link each summary to its related coarse chunk
#H Ingest the summaries into the vector store
#I Ingest the coarse chunks into the document store

In [ ]:
# COMMENT: the code above is similar to when ingesting child chunks, but it is slower because of the summarization step
# which invokes the LLM.
# The processing can be speeded up by parallelizing the outer for loop on the destination urls.

### Performing a search on granular information

### Searching summary embeddings

The retriever searches compact summary vectors but returns the full source chunks. Directly searching the summary collection shows the focused representation, while the retriever output provides the broader context needed for synthesis.


In [ ]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall travel")

In [ ]:
len(retrieved_docs)

In [ ]:
retrieved_docs

### Comparing with direct semantic search on summaries

In [ ]:
summary_docs_only =  summaries_collection.similarity_search(
    "Cornwall Travel")

In [ ]:
len(summary_docs_only)

In [ ]:
summary_docs_only

In [ ]:
# COMMENT: a direct search on summaries retrieves denser information, but it is missing out on useful details. 
# However, you might consider using the summaries directly if after testing they prove adequate.

### Hypothetical question embeddings

Sometimes a user's question is semantically closer to a question that a document can answer than to the document's original wording. Hypothetical question embeddings exploit this by generating likely questions for each chunk and embedding those questions.

At query time, the user question can match one of these generated questions, which then points back to the original source chunk.


## Embedding hypothetical questions with MultiVectorRetriever

In [ ]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from local_html import LocalHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid
from typing import List
from pydantic import BaseModel, Field

### Setting up the Multi vector retriever (same as when embedding summaries)

In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A

hypothetical_questions_collection = Chroma( #B
    collection_name="uk_hypothetical_questions",
    embedding_function=get_embeddings_model(),
)

hypothetical_questions_collection.reset_collection() #C

doc_byte_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=hypothetical_questions_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host parent coarse chunks
#E Retriever to link parent coarse chunks to child granular chunks

### Setting up the chain to generate hypothetical questions

### Structured question generation

The `HypotheticalQuestions` schema asks the LLM to return a typed list of generated questions. Structured output keeps the downstream ingestion code simple because it can rely on receiving a list rather than parsing free-form text.


In [ ]:
class HypotheticalQuestions(BaseModel):
    """A list of hypotetical questions for given text."""

    questions: List[str] = Field(..., description="List of hypothetical questions for given text")

In [ ]:
llm_with_structured_output = get_chat_model().with_structured_output(
        HypotheticalQuestions
)

In [ ]:
hypothetical_questions_chain = (
    {"document_text": lambda x: x.page_content} #A
    | ChatPromptTemplate.from_template( #B
        "Generate a list of exactly 4 hypothetical questions that the below text could be used to answer:\n\n{document_text}"
    )
    | llm_with_structured_output #C
    | (lambda x: x.questions) #D
)

#A Grab the text content from the document
#B Instantiate a prompt asking to generate 4 hypothetical questions on the provided text
#C Invoke the LLM configured to return an object containing the questions as a typed list of strings
#D Grab the list of questions from the response

### Ingesting the coarse chunks and related hypothetical questions into doc and vector store

### Ingesting hypothetical questions

For each coarse chunk, the chain generates several questions that the chunk could answer. Each generated question is stored as a separate document in the vector store, with metadata linking it back to the original coarse chunk.


In [ ]:
for destination_url in uk_destination_urls:
    html_loader = LocalHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_hypothetical_questions = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E
        
        coarse_chunk_id = coarse_chunks_ids[i]
            
        hypothetical_questions = hypothetical_questions_chain.invoke(
            coarse_chunk) #F
        hypothetical_questions_docs = [Document(
            page_content=question, metadata={doc_key: coarse_chunk_id})
                    for question 
                    in hypothetical_questions] #G

        all_hypothetical_questions.extend(hypothetical_questions_docs)

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_hypothetical_questions) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination 
#C transform HTML docs into clean text docs
#D Split the destination content into coarse chunks
#E Iterate over the coarse chunks
#F Generate a list of hypothetical questions for the coarse chunk thorugh the question generation chain
#G Link each hypothetical question to its related coarse chunk
#H Ingest the hypothetical questions into the vector store
#I Ingest the coarse chunks into the document store

### Performing a search on granular information

### Searching hypothetical questions

The retriever searches over generated question embeddings and returns the linked source chunks. This improves recall when the wording of the user's question differs from the wording of the source document.


In [ ]:
retrieved_docs = multi_vector_retriever.invoke(
    "How can you go to Brighton from London?")

In [ ]:
len(retrieved_docs)

In [ ]:
retrieved_docs

### Inspecting possible questions matching our question through semantic search

### Direct hypothetical-question comparison

A direct search on the hypothetical-question collection reveals which generated questions matched the user's query. The full retriever is more useful for RAG because it maps those matching questions back to the original context.


In [ ]:
hypothetical_question_docs_only = hypothetical_questions_collection.similarity_search(
    "How can you go to Brighton from London?")

In [ ]:
len(hypothetical_question_docs_only)

In [ ]:
hypothetical_question_docs_only

# Granular chunk expansion with MultiVectorRetriever

In [ ]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from local_html import LocalHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

### Setting up the Multi vector retriever

In [ ]:
granular_chunk_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500) #A

granular_chunks_collection = Chroma( #B
    collection_name="uk_granular_chunks",
    embedding_function=get_embeddings_model(),
)

granular_chunks_collection.reset_collection() #C

expanded_chunk_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=granular_chunks_collection,
    byte_store=expanded_chunk_store
)
#A Splitter to generate granular chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host expanded chunks
#E Retriever to link parent coarse chunks to child granular chunks

### Ingesting granular and expanded chunks into doc and vector store

In [ ]:
for destination_url in uk_destination_urls:
    html_loader = LocalHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    granular_chunks = granular_chunk_splitter.split_documents(
        text_docs) #D

    expanded_chunk_store_items = []
    for i, granular_chunk in enumerate(
        granular_chunks): #E

        this_chunk_num = i #F
        previous_chunk_num = i-1 #F
        next_chunk_num = i+1 #F
        
        if i==0: #F
            previous_chunk_num = None
        elif i==(len(granular_chunks)-1): #F
            next_chunk_num = None

        expanded_chunk_text = "" #G
        if previous_chunk_num: #G
            expanded_chunk_text += granular_chunks[
                previous_chunk_num].page_content
            expanded_chunk_text += "\n"

        expanded_chunk_text += granular_chunks[
            this_chunk_num].page_content #G
        expanded_chunk_text += "\n"

        if next_chunk_num: #G
            expanded_chunk_text += granular_chunks[
                next_chunk_num].page_content
            expanded_chunk_text += "\n"

        expanded_chunk_id = str(uuid.uuid4()) #H
        expanded_chunk_doc = Document(
            page_content=expanded_chunk_text) #I

        expanded_chunk_store_item = (expanded_chunk_id, 
                                     expanded_chunk_doc)
        expanded_chunk_store_items.append(
            expanded_chunk_store_item)

        granular_chunk.metadata[
            doc_key] = expanded_chunk_id #J
            
    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        granular_chunks) #K
    multi_vector_retriever.docstore.mset(
        expanded_chunk_store_items) #L

#A Loader for one destination
#B Documents of one destination 
#C transform HTML docs into clean text docs
#D Split the destination content into granular chunks
#E Iterate over the granular chunks
#F determine the index of the current chunk and its previous and next chunks
#G Assemble the text of the expanded chunk by including the previous and next chunk
#H Generate the ID of the expanded chunk
#I Create the expanded chunk document
#J Link each granular chunk to its related expanded chunk
#K Ingest the granular chunks into the vector store
#L Ingest the expanded chunks into the document store

### Performing a search on granular information

In [ ]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall Ranger")

In [ ]:
len(retrieved_docs)

In [ ]:
retrieved_docs[0]

### Comparing with direct semantic search on granular chunks

In [ ]:
child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")

In [ ]:
len(child_docs_only)

In [ ]:
child_docs_only[0]

In [ ]:
# COMMENT: the expanded chunk has more useful context